# Tantivy GBIF Taxon Index (CSV → local index)

This notebook builds a **local Tantivy index** where **1 `taxonID` = 1 document**, optimized for your pipeline:
1) Extract candidate taxon-like terms from an abstract/full text
2) Look them up in the Tantivy index (BM25 + edge-prefix fallback)
3) Return the best-matching taxon records

We add **edge n-grams (prefixes)** of length **3–10** for:
- `canonicalName`
- `genericName`
- `vernaculars_named`

…as separate fields (best practice) so exact/phrase matches can still dominate ranking.


## 1) Configure paths and take a quick look at the CSV


In [ ]:
from pathlib import Path
import pandas as pd

import os
import re
from typing import Optional

import tantivy
from tantivy import SchemaBuilder, Index, Document, Query, Occur


In [ ]:
#.env stuff
CSV_PATH = Path("data/gbif/curated/gbif_curated.csv") 
INDEX_DIR = Path("data/retrieval/tantivy_gbif_taxa_index")


EDGE_MIN = 3
EDGE_MAX = 10

cpu = os.cpu_count() or 8
NUM_THREADS = min(6, max(2, cpu - 2))   # cap at 6, leave some CPU for the OS
HEAP_SIZE_BYTES = 1_500_000_000
CHUNKSIZE = 50_000


In [ ]:
status_sel = ['accepted', 'synonym' 'homotypic synonym' ,'heterotypic synonym' 'proparte synonym']
rank_sel = ['kingdom', 'phylum', 'class', 'order', 'genus', 'species'  ]
subset = ['taxonID', 
       'scientificName', 'scientificNameAuthorship',
       'canonicalName', 'genericName', 'specificEpithet',
       'infraspecificEpithet', 'taxonRank', 'taxonomicStatus', 'nomenclaturalStatus',
        'kingdom', 'phylum', 'class', 'order', 'family',
       'genus',  'vernaculars_named', 'curated_description',
       'species_id', 'livingPeriod_curated', 'lifeForm_curated',
       'habitat_curated', 'marine_curated', 'freshwater_curated',
       'terrestrial_curated', 'extinct_curated', 'hybrid_curated',
       'ageInDays_curated', 'sizeInMillimeter_curated', 'massInGram_curated'
       ]

In [ ]:
df = pd.read_csv(CSV_PATH)
df = df[(df.taxonRank.isin(rank_sel)) & ((df.taxonomicStatus.isin(status_sel)))][subset]

display(df.head())
print("Number of rows:", len(df))
print("Columns:", list(df.columns))


## 2) Field design: analyzed vs not analyzed

### Analyzed (tokenized) fields
These are used for full-text matching. Tantivy's default tokenizer splits on punctuation/whitespace and lowercases.
- `canonical_name`
- `scientific_name`
- `generic_name`
- `vernaculars`
- `name_text` (a combined catch-all field)

### Prefix (edge n-gram) fields
We create separate fields with prefixes (3–10) so partial strings like `panthe` can match `panthera`.
- `canonical_edge`
- `generic_edge`
- `vernacular_edge`

### Keyword-ish fields (not split)
For exact filtering/grouping, we index them with the `raw` tokenizer (no splitting).
- `taxon_rank`, `taxonomic_status`, `kingdom`, `phylum`, `class`, `order`, `family`, `genus`, `source`

### Stored-only fields
Large/noisy fields are useful to retrieve after a match, but not useful for matching. We'll try to store-only.
(If your tantivy-py version doesn't support `indexed=False` for text fields, you'll either upgrade or drop these.)


## 3) Helpers
We:
- normalize keyword values
- split vernacular names
- generate **edge n-grams** (prefixes) 3–10


In [ ]:
def norm_keyword(x: Optional[str]) -> Optional[str]:
    """Normalize keyword-ish fields for exact filtering."""
    if x is None:
        return None
    s = str(x).strip().lower()
    if not s or s == "nan":
        return None
    s = re.sub(r"\s+", "_", s)  # spaces -> underscores
    return s

def split_vernaculars(x: Optional[str]) -> list[str]:
    """Split vernaculars_named which is often delimiter-joined."""
    if x is None:
        return []
    s = str(x).strip()
    if not s or s == "nan":
        return []
    parts = re.split(r"[|;,\n\t]+", s)
    return [p.strip() for p in parts if p and p.strip()]

def to_int_or_none(x) -> Optional[int]:
    try:
        if x is None or (hasattr(pd, "isna") and pd.isna(x)):
            return None
        return int(float(x))
    except Exception:
        return None

def to_bool01(x) -> int:
    if x is None or (hasattr(pd, "isna") and pd.isna(x)):
        return 0
    if isinstance(x, bool):
        return 1 if x else 0
    s = str(x).strip().lower()
    return 1 if s in ("1", "true", "t", "yes", "y") else 0

def edge_ngrams(text: Optional[str], min_n: int = EDGE_MIN, max_n: int = EDGE_MAX) -> list[str]:
    """Generate edge n-grams (prefixes) for each alphanumeric token."""
    if text is None:
        return []
    s = str(text).strip().lower()
    if not s or s == "nan":
        return []

    tokens = re.findall(r"[0-9a-z]+", s)
    grams = set()
    for tok in tokens:
        if len(tok) < min_n:
            continue
        upper = min(max_n, len(tok))
        for n in range(min_n, upper + 1):
            grams.add(tok[:n])
    return sorted(grams)

def make_name_text(row: dict) -> str:
    """Catch-all searchable blob: names + hierarchy + vernaculars."""
    parts = []
    for k in [
        "canonicalName", "scientificName", "genericName",
        "specificEpithet", "infraspecificEpithet",
        "kingdom", "phylum", "class", "order", "family", "genus",
    ]:
        v = row.get(k)
        if v is not None and not pd.isna(v):
            s = str(v).strip()
            if s and s.lower() != "nan":
                parts.append(s)
    parts.extend(split_vernaculars(row.get("vernaculars_named")))
    return " | ".join(parts)

def first(doc, field, default=None):
    try:
        vals = doc[field]          # vals is a list
        return vals[0] if vals else default
    except KeyError:
        return default


## 4) Build the Tantivy schema

We’ll build a schema with:
- `taxon_id` indexed+stored (so you can retrieve it)
- name fields analyzed
- edge prefix fields indexed-only
- keyword-ish fields using the `raw` tokenizer

Compatibility note: some tantivy-py versions support `indexed=` for text fields; some don’t.
We use a helper `add_text_field_safe()` to gracefully handle this.


In [ ]:
def build_schema() -> tantivy.Schema:
    sb = SchemaBuilder()

    # Primary ID
    sb.add_integer_field("taxon_id", indexed=True, stored=True, fast=True)

    # Optional numeric fields
    sb.add_integer_field("species_id", indexed=False, stored=True, fast=True)

    # Analyzed name fields (tokenized)
    sb.add_text_field("canonical_name", stored=True, tokenizer_name="default")
    sb.add_text_field("scientific_name", stored=True, tokenizer_name="default")
    sb.add_text_field("generic_name", stored=True, tokenizer_name="default")
    sb.add_text_field("vernaculars", stored=True, tokenizer_name="default")
    sb.add_text_field("name_text", tokenizer_name="default")  # not stored

    # Edge prefix fields (indexed; not stored)
    sb.add_text_field("canonical_edge", tokenizer_name="default")
    sb.add_text_field("scientific_edge", tokenizer_name="default")  # <-- if you added scientific edge grams
    sb.add_text_field("generic_edge", tokenizer_name="default")
    sb.add_text_field("vernacular_edge", tokenizer_name="default")

    # Keyword-ish fields (exact-ish) using raw tokenizer
    sb.add_text_field("taxon_rank", stored=True, tokenizer_name="raw")
    sb.add_text_field("taxonomic_status", stored=True, tokenizer_name="raw")
    for fld in ["kingdom", "phylum", "class", "order", "family", "genus", "source"]:
        sb.add_text_field(fld, stored=True, tokenizer_name="raw")

    # Flags / numeric curated
    for fld in ["marine", "freshwater", "terrestrial", "extinct", "hybrid"]:
        sb.add_integer_field(fld, indexed=False, stored=True, fast=True)
    for fld in ["age_days", "size_mm", "mass_g"]:
        sb.add_integer_field(fld, indexed=False, stored=True, fast=True)

    # OPTION B: keep big/noisy fields as stored text
    # (They WILL be indexed too in this API — but you just won't query them.)
    sb.add_text_field("curated_description", stored=True, tokenizer_name="default")
    sb.add_text_field("scientific_name_authorship", stored=True, tokenizer_name="default")
    sb.add_text_field("lifeform_curated", stored=True, tokenizer_name="default")
    sb.add_text_field("habitat_curated", stored=True, tokenizer_name="default")
    sb.add_text_field("livingperiod_curated", stored=True, tokenizer_name="default")
    sb.add_text_field("nomenclatural_status", stored=True, tokenizer_name="default")

    return sb.build()


## 5) Indexing: CSV → Tantivy documents

We stream the CSV in chunks. For each row:
- build a Tantivy `Document`
- add name fields
- generate edge prefixes for the three requested name sources and add them into separate fields
- add keyword-ish fields and numeric flags
- commit once at the end

Tip: for a much larger dataset, you can increase `heap_size` and chunk size, and keep `num_threads` near CPU cores.


In [ ]:
def build_index(csv_path: Path, index_dir: Path) -> None:
    schema = build_schema()
    index_dir.mkdir(parents=True, exist_ok=True)

    # Create a persistent index on disk
    index = Index(schema=schema, path=str(index_dir))

    writer = index.writer(heap_size=HEAP_SIZE_BYTES, num_threads=NUM_THREADS)

    for chunk in pd.read_csv(csv_path, chunksize=CHUNKSIZE):
        for row in chunk.to_dict(orient="records"):
            taxon_id = to_int_or_none(row.get("taxonID"))
            if taxon_id is None:
                continue

            doc = Document()
            doc.add_integer("taxon_id", taxon_id)

            # Name fields + edge prefixes
            cn = row.get("canonicalName")
            if cn is not None and not pd.isna(cn):
                cn = str(cn)
                doc.add_text("canonical_name", cn)
                for g in edge_ngrams(cn):
                    doc.add_text("canonical_edge", g)


            sn = row.get("scientificName")
            if sn is not None and not pd.isna(sn):
                sn = str(sn)
                doc.add_text("scientific_name", sn)
                for g in edge_ngrams(sn):
                    doc.add_text("scientific_edge", g)


            gn = row.get("genericName")
            if gn is not None and not pd.isna(gn):
                gn = str(gn)
                doc.add_text("generic_name", gn)
                for g in edge_ngrams(gn):
                    doc.add_text("generic_edge", g)

            vern_list = split_vernaculars(row.get("vernaculars_named"))
            for v in vern_list:
                doc.add_text("vernaculars", v)
                for g in edge_ngrams(v):
                    doc.add_text("vernacular_edge", g)

            # Catch-all (helps recall if you want it)
            doc.add_text("name_text", make_name_text(row))

            # Keyword-ish / filters
            rank = norm_keyword(row.get("taxonRank"))
            if rank:
                doc.add_text("taxon_rank", rank)

            status = norm_keyword(row.get("taxonomicStatus"))
            if status:
                doc.add_text("taxonomic_status", status)

            for fld in ["kingdom", "phylum", "class", "order", "family", "genus", "source"]:
                v = norm_keyword(row.get(fld))
                if v:
                    doc.add_text(fld, v)

            # Stored-only details
            a = row.get("scientificNameAuthorship")
            if a is not None and not pd.isna(a):
                doc.add_text("scientific_name_authorship", str(a))

            cd = row.get("curated_description")
            if cd is not None and not pd.isna(cd):
                doc.add_text("curated_description", str(cd))

            for src_col, dst in [
                ("lifeForm_curated", "lifeform_curated"),
                ("habitat_curated", "habitat_curated"),
                ("livingPeriod_curated", "livingperiod_curated"),
                ("nomenclaturalStatus", "nomenclatural_status"),
            ]:
                vv = row.get(src_col)
                if vv is not None and not pd.isna(vv):
                    doc.add_text(dst, str(vv))

            # Numeric extras
            sid = to_int_or_none(row.get("species_id"))
            if sid is not None:
                doc.add_integer("species_id", sid)


            # Flags
            doc.add_integer("marine", to_bool01(row.get("marine_curated")))
            doc.add_integer("freshwater", to_bool01(row.get("freshwater_curated")))
            doc.add_integer("terrestrial", to_bool01(row.get("terrestrial_curated")))
            doc.add_integer("extinct", to_bool01(row.get("extinct_curated")))
            doc.add_integer("hybrid", to_bool01(row.get("hybrid_curated")))

            # Curated numeric
            age = to_int_or_none(row.get("ageInDays_curated"))
            if age is not None:
                doc.add_integer("age_days", age)
            size = to_int_or_none(row.get("sizeInMillimeter_curated"))
            if size is not None:
                doc.add_integer("size_mm", size)
            mass = to_int_or_none(row.get("massInGram_curated"))
            if mass is not None:
                doc.add_integer("mass_g", mass)

            writer.add_document(doc)

    writer.commit()
    writer.wait_merging_threads()
    index.reload()
    print(f"✅ Index built at: {index_dir.resolve()}")


## 7) (Optional) Candidate extraction from an abstract

This is a lightweight extractor you can refine later:
- capture binomials like `Genus species`
- capture genus-only candidates (capitalized words)

Then you feed the extracted list into `search_taxa()`.


In [ ]:
import shutil

shutil.rmtree(INDEX_DIR, ignore_errors=True)


In [ ]:
build_index(CSV_PATH, INDEX_DIR)


In [ ]:
index = tantivy.Index.open(str(INDEX_DIR))
index.reload()
searcher = index.searcher()

In [ ]:
import tantivy

def first(doc, field, default=None):
    try:
        vals = doc[field]          # vals is a list
        return vals[0] if vals else default
    except KeyError:
        return default

query_str = '"mammal" OR panthe' # OR panthe

q_exact = Query.boost_query(
    index.parse_query(
        query_str,
        ["canonical_name", "scientific_name", "generic_name", "vernaculars", "name_text"],
    ),
    3.0,
)

q_edge = Query.boost_query(
    index.parse_query(
        query_str,
        ["canonical_edge", "scientific_edge", "generic_edge", "vernacular_edge"],
    ),
    1.0,
)

q_desc = Query.boost_query(
    index.parse_query(query_str, ["curated_description"]),
    0.2,   # very low, so it helps recall but doesn't dominate
)

q = Query.disjunction_max_query([q_exact, q_edge, q_desc], tie_breaker=0.1)

hits = searcher.search(q, 10).hits

for score, addr in hits:
    doc = searcher.doc(addr)
    # print(doc)
    # print(score, doc.get("taxon_id"), doc.get("canonical_name"), doc.get("taxon_rank"))
    
    print(score, '|', 
          first(doc, "taxon_id"), '|',
          first(doc,"taxon_rank") , "|",
          first(doc, "taxonomic_status"), "|",
          first(doc, "scientific_name"), '|', 
          first(doc, "canonical_name"), '|', 
          )


In [ ]:
doc

In [7]:
import re

STOP = {
    "We","I","Our","The","A","An","This","That","These","Those","In","On","At","By","For","From",
    "With","Without","As","And","Or","But","However","Moreover","Therefore","Thus","Results","Methods",
    "Introduction","Discussion","Conclusion","Study","Table","Figure","Data","Analysis"
}

# Optional: expand with journals’ boilerplate words you see often.

def extract_taxon_candidates(text: str, max_terms: int = 80) -> list[str]:
    # 1) High-precision binomials: Genus species (species epithet lowercase, allow hyphen)
    binomials = re.findall(r"\b([A-Z][a-z]{3,})\s+([a-z][a-z-]{1,})\b", text)
    binomial_terms = [f"{g} {s}" for g, s in binomials]

    # 2) Genus + qualifier patterns: "Genus sp.", "Genus spp.", "Genus cf.", etc.
    genus_qual = re.findall(
        r"\b([A-Z][a-z]{3,})\s+(sp\.|spp\.|cf\.|aff\.|nr\.|gen\.|nov\.|subsp\.|var\.)\b",
        text
    )
    genus_terms = [g for g, _ in genus_qual]

    # 3) De-dup, filter stopwords/acronyms
    seen = set()
    out = []
    for t in binomial_terms + genus_terms:
        if t in seen:
            continue
        # filter obvious non-taxa
        head = t.split()[0]
        if head in STOP:
            continue
        if head.isupper():  # acronyms
            continue
        seen.add(t)
        out.append(t)
        if len(out) >= max_terms:
            break

    return out


In [8]:
abstract = """
Does illegal hunting affect density and behaviour of African grassland birds? A case study on ostrich (Struthio camelus)	Widespread bushmeat hunting represents one of the major threats to many mammals and birds in Africa. We studied the influence of illegal bushmeat hunting on large grassland birds in the Serengeti National Park (SNP) and adjoining protected areas, by using the ostrich (Struthio camelus) as a case study. First, we documented illegal hunting of both small and large birds by using a questionnaire in the villages on the western and eastern side of the SNP. Second, we studied the effect of illegal hunting on density by driving 4,659 km of transects inside SNP and on the adjacent protected areas, where the data were analysed by DISTANCE sampling. Last, we used flight initiation distance (FID, i.e. the distance between an approaching predator (human) and prey when flight is started), to assess possible impacts on behaviour from illegal hunting. We found that people from the western side of the SNP admitted to hunting both small and large grassland birds, and collect ostrich feathers and eggs. Although the Maasai also hunted small birds, only ostrich feathers and eggs of the large grassland birds were used. Surprisingly, we found no significant differences in densities between the SNP and adjoining partially protected areas, but ostriches had longer FID to an approaching human outside the SNP. Currently illegal hunting does not appear to affect the ostrich population, but given the extensive use of birds for consumption more awareness educational programs accompanied by provision of agricultural incentives within the protected areas are needed.
"""
candidates = extract_taxon_candidates(abstract)
print(len(candidates))
print(candidates)

7
['Does illegal', 'African grassland', 'Struthio camelus', 'Widespread bushmeat', 'Although the', 'Maasai also', 'Currently illegal']


In [1]:
from retrieval import IndexConfig, FilterConfig, build_index, TaxonSearcher, extract_taxon_candidates


In [2]:
cfg = IndexConfig.from_env()


In [3]:
cfg

IndexConfig(csv_path=PosixPath('data/gbif/curated/gbif_curated.csv'), index_dir=PosixPath('data/retrieval/tantivy_gbif_curated'), edge_min=3, edge_max=10, num_threads=6, heap_size_bytes=1500000000, chunksize=50000)

In [4]:
FilterConfig

retrieval.config.FilterConfig

In [ ]:
build_index(cfg, FilterConfig())

In [5]:
searcher = TaxonSearcher(cfg.index_dir)
results = searcher.search("Philipsalta", limit=40)


In [6]:
results

[SearchResult(score=205.43426513671875, taxon_id=7508233, taxon_rank='genus', taxonomic_status='accepted', scientific_name='Philipsalta Lee, Marshall & Hill, 2016', canonical_name='Philipsalta', document=Document(canonical_name=[Philipsalt],class=[insecta],curated_description=[descriptio],extinct=[0],family=[cicadidae],freshwater=[0],generic_name=[Philipsalt],genus=[philipsalt],hybrid=[0],kingdom=[animalia],marine=[0],order=[hemiptera],phylum=[arthropoda],scientific_name=[Philipsalt],scientific_name_authorship=[Lee, Marsh],taxon_id=[7508233],taxon_rank=[genus],taxonomic_status=[accepted],terrestrial=[0])),
 SearchResult(score=192.80108642578125, taxon_id=12263532, taxon_rank='species', taxonomic_status='accepted', scientific_name='Philipsalta exilis Lee & Marshall, 2023', canonical_name='Philipsalta exilis', document=Document(canonical_name=[Philipsalt],class=[insecta],curated_description=[descriptio],extinct=[0],family=[cicadidae],freshwater=[0],generic_name=[Philipsalt],genus=[philip

In [ ]:
abstract = "..."
candidates = extract_taxon_candidates(abstract)